## Exploración inicial
Los datasets que vamos a evaluar tienen un gran número de filas (38 millones). Por este motivo durante la epxloración inicial usaremos la librería Polars en vez de Pandas. Polars también funciona con dataframes.

In [48]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

In [15]:
import polars as pl
from src.data_loader import load_lazy, load_columns
from src.config import CIC_TRAIN_PATH, CIC_TEST_PATH

Cargamos los datasets iniciales:
 * cic_iot_2023_train.parquet: "Datos de entrenamiento"
 * cic_iot_2023_test.parquet: "Datos de testing"

En la mayoría de casos, las funciones de polars son similares a las de pandas (métodos como *.head()* o *.describe()* realizan la misma función). La diferencia crucial es que *polars* no carga todos los datos a memoria directamente. Cuando queramos ver las filas que hemos seleccionado, tendremos que llamar a *.collect()* o *.fetch()*.

In [19]:
train_df = load_lazy(CIC_TRAIN_PATH)
test_df = load_lazy(CIC_TEST_PATH)

print(train_df)
train_df.head().collect()

naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

Parquet SCAN [C:/Users/marco/Bootcamp_projects/cibersecurity_ml_project/datasets/raw/cic_iot_2023_train.parquet]
PROJECT */42 COLUMNS
ESTIMATED ROWS: 30806432


Header_Length,Protocol Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,cwr_flag_number,ack_count,syn_count,fin_count,rst_count,HTTP,HTTPS,DNS,Telnet,SMTP,SSH,IRC,TCP,UDP,DHCP,ARP,ICMP,IGMP,IPv,LLC,Tot sum,Min,Max,AVG,Std,Tot size,IAT,Number,Variance,Label,attack_class,label
f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,i64
20.0,6,64.0,48998.878505,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,6000.0,60.0,60.0,60.0,0.0,60.0,0.00002,100.0,0.0,"""DDoS-TCP_Flood""","""DDoS""",1
3.92,17,63.36,1257.669911,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.49,0.0,0.01,0.0,0.0,0.99,0.99,92346.0,60.0,1514.0,923.46,582.536831,923.46,0.000795,100.0,339349.16,"""DDoS-UDP_Fragmentation""","""DDoS""",1
0.0,1,64.0,160762.897662,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,6000.0,60.0,60.0,60.0,0.0,60.0,0.000006,100.0,0.0,"""DDoS-ICMP_Flood""","""DDoS""",1
20.0,6,64.0,19372.333841,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,6000.0,60.0,60.0,60.0,0.0,60.0,0.000052,100.0,0.0,"""DoS-TCP_Flood""","""DoS""",1
20.0,6,64.0,35638.57592,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.01,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,6000.0,60.0,60.0,60.0,0.0,60.0,0.000028,100.0,0.0,"""DDoS-SynonymousIP_Flood""","""DDoS""",1


Echamos un vistazo a las features con las que vamos a trabajar.

In [47]:
train_df.collect_schema()

Schema([('Header_Length', Float64),
        ('Protocol Type', Int64),
        ('Time_To_Live', Float64),
        ('Rate', Float64),
        ('fin_flag_number', Float64),
        ('syn_flag_number', Float64),
        ('rst_flag_number', Float64),
        ('psh_flag_number', Float64),
        ('ack_flag_number', Float64),
        ('ece_flag_number', Float64),
        ('cwr_flag_number', Float64),
        ('ack_count', Float64),
        ('syn_count', Float64),
        ('fin_count', Float64),
        ('rst_count', Float64),
        ('HTTP', Float64),
        ('HTTPS', Float64),
        ('DNS', Float64),
        ('Telnet', Float64),
        ('SMTP', Float64),
        ('SSH', Float64),
        ('IRC', Float64),
        ('TCP', Float64),
        ('UDP', Float64),
        ('DHCP', Float64),
        ('ARP', Float64),
        ('ICMP', Float64),
        ('IGMP', Float64),
        ('IPv', Float64),
        ('LLC', Float64),
        ('Tot sum', Float64),
        ('Min', Float64),
        ('Max', Fl

Las columnas que vamos a predecir son aquellas que tienen el nombre de cada tipo de ataque. En este dataset hay 3 targets:
* *Label*: El tipo de subataque exacto
* *attack_class*: El grupo de ataque general
* *label*: Etiqueta binaria de si el paquete es normal (0) o malicioso (1)

In [27]:
train_dist = (train_df.group_by("Label").len().sort("len", descending=True).collect())
print(train_dist)

shape: (32, 2)
┌─────────────────────────┬─────────┐
│ Label                   ┆ len     │
│ ---                     ┆ ---     │
│ str                     ┆ u32     │
╞═════════════════════════╪═════════╡
│ DDoS-ICMP_Flood         ┆ 5760907 │
│ DDoS-UDP_Flood          ┆ 4328947 │
│ DDoS-TCP_Flood          ┆ 3597645 │
│ DDoS-SYN_Flood          ┆ 3249209 │
│ DDoS-SynonymousIP_Flood ┆ 2776604 │
│ …                       ┆ …       │
│ SqlInjection            ┆ 4224    │
│ XSS                     ┆ 3081    │
│ Backdoor_Malware        ┆ 2579    │
│ Recon-PingSweep         ┆ 1801    │
│ Uploading_Attack        ┆ 998     │
└─────────────────────────┴─────────┘


In [28]:
train_dist = (train_df.group_by("attack_class").len().sort("len", descending=True).collect())
print(train_dist)

shape: (8, 2)
┌──────────────┬──────────┐
│ attack_class ┆ len      │
│ ---          ┆ ---      │
│ str          ┆ u32      │
╞══════════════╪══════════╡
│ DDoS         ┆ 20574495 │
│ DoS          ┆ 6275648  │
│ Mirai        ┆ 2106431  │
│ Benign       ┆ 878501   │
│ Recon        ┆ 552177   │
│ Spoofing     ┆ 388807   │
│ Web-based    ┆ 19939    │
│ BruteForce   ┆ 10434    │
└──────────────┴──────────┘


In [29]:
train_binary = train_df.group_by("label").len().collect()
print(train_binary)

shape: (2, 2)
┌───────┬──────────┐
│ label ┆ len      │
│ ---   ┆ ---      │
│ i64   ┆ u32      │
╞═══════╪══════════╡
│ 1     ┆ 29927931 │
│ 0     ┆ 878501   │
└───────┴──────────┘


Con esta exploración inicial de los labels, podemos ver que el dataset presenta un gran problema de desbalance de datos.

Los ataques como "DDoS" están sobre representados con 20~ millones filas mientras que otros ataques como "BruteForce" o "Web-based" apenas llegan a los 20.000 ejemplos.
* Los labels binarios también tienen un problema de desbalanceo, con 30~ millones de los ejemplos perteneciendo a tráfico malicioso y solo 900.000~ siendo normales.

Por otra parte también hemos visto que hay muchas features (42). Utilizando un poco de conocimiento específico, seleccionamos columnas que, a primera vista, pueden parecer que ya han sido tratadas.
* Elegimos "Protocol Type" y después una serie de columnas que representan un protocolo cada una ("HTTP", "TCP", "UDP", ...)

In [ ]:
cols = ["Protocol Type", "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IGMP", "IPv", "LLC"]

df_feat = load_columns(CIC_TRAIN_PATH, cols)
df_feat.head(10).collect()

Protocol Type,HTTP,HTTPS,DNS,Telnet,SMTP,SSH,IRC,TCP,UDP,DHCP,ARP,ICMP,IGMP,IPv,LLC
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.49,0.0,0.01,0.0,0.0,0.99,0.99
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
6,0.0,0.01,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
17,0.0,0.0,0.02,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0
6,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0


Inicialmente podemos pensar que, han aplicado *One-Hot Encoding* ya que algunas filas tienen "Protocol Type=6" y "TCP=1.0", "Protocol Type=17" y "UDP=1.0", "Protocol Type=1" y "ICMP=1.0".

Sin embargo, también vemos que los valores no se limitan a 1.0 y 0.0, sino que algunas filas tienen valores como 0.49 o o 0.01.
Sumado a esto, en Internet hay una gran cantidad de protocolos. Ahora vamos a comprobar qué tipo de protocolos exactamente está representando la columna de "Protocol Type".

In [32]:
protocol_counts = (train_df.group_by("Protocol Type").len().sort("len", descending=True).collect())
protocol_counts

Protocol Type,len
i64,u32
6,15290666
17,8004987
1,6117567
47,1382144
0,11065
2,3


Solo 6 valores distintos, pero reconocibles. Los protocolos tienen un número asignado como parte de un estándar universal de Internet. 

Estos números suelen ir en la cabecera de los paquetes IP y sirven como meta-datos que explican el protocolo que va dentro del paquete. En este caso el mapeo sería así: 

| Valor | Protocolo | Capa OSI | Descripción |
|------:|-----------|----------|-------------|
| 0     | HOPOPT / no especificado | Capa 3 (Red) | Valor reservado o no definido |
| 1     | ICMP | Capa 3 (Red) | Protocolo de mensajes de control y diagnóstico de red |
| 2     | IGMP | Capa 3 (Red) | Protocolo de gestión de grupos de Internet |
| 6     | TCP | Capa 4 (Transporte) | Protocolo de transporte fiable orientado a conexión |
| 17    | UDP | Capa 4 (Transporte) | Protocolo de transporte no orientado a conexión |
| 47    | GRE | Capa 3 / Túneles | Encapsulación de enrutamiento genérico (túneles de red) |


Con esta información, seleccionamos las columnas que nos interesan.

In [39]:
cols = ["Protocol Type", "TCP", "UDP", "ICMP", "IGMP"]

df_feat = load_columns(CIC_TRAIN_PATH, cols)
df_feat.head(10).collect()

Protocol Type,TCP,UDP,ICMP,IGMP
i64,f64,f64,f64,f64
6,1.0,0.0,0.0,0.0
17,0.0,0.49,0.0,0.0
1,0.0,0.0,1.0,0.0
6,1.0,0.0,0.0,0.0
6,1.0,0.0,0.0,0.0
6,1.0,0.0,0.0,0.0
17,0.0,1.0,0.0,0.0
6,1.0,0.0,0.0,0.0
17,0.0,1.0,0.0,0.0


En primer lugar, no existe una columna que mapee el número de protocolo de algunos como GRE (47) o el 0. Adicionalmente, podemos ver que algunas columnas tienen diferentes como 0.49 y 1.0 para el mismo valor de "Protocol Type=17". 

Debido a esto, podemos concluir que columnas como "TCP" o "UDP" no son columnas dummies para "Protocol Type", sino que representan otro tipo de información.

No hay información sobre lo que cada columna representa exactamente. Imagino que será un tipo de indicador "soft", que explica la proporción del tráfico que pertenece a ese protocolo durante la conexión.

In [42]:
cols = ["Time_To_Live", "Header_Length"]

df_feat = load_columns(CIC_TRAIN_PATH, cols)
df_feat.head(2).collect()

Time_To_Live,Header_Length
f64,f64
64.0,20.0
63.36,3.92


Esto también explica por qué algunos campos como el "Time_To_Live" o "Header_Length" tienen valores decimales cuando no tendría sentido que los tuviese si cada fila representase un paquete únicamente.

Cada fila representa probablemente una conexión y tiene información sobre el tráfico asociada a ella. Podemos visualizar algunos campos relacionados con esto:

In [ ]:
numeric_cols = ["Header_Length", "Time_To_Live", "Rate", "Tot sum", "Min", "Max", "AVG", "Std", "Tot size", "IAT", "Number", "Variance"]
df_num = load_columns(CIC_TRAIN_PATH, numeric_cols)
df_num.collect().describe()

statistic,Header_Length,Time_To_Live,Rate,Tot sum,Min,Max,AVG,Std,Tot size,IAT
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",3.0806432e7,3.0806432e7,3.0806432e7,3.0806432e7,3.0806432e7,3.0806432e7,3.0806432e7,3.0806432e7,3.0806432e7,3.0806432e7
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",12.432862,67.010415,27490.945132,12028.897164,84.341171,253.315368,146.994667,49.670498,146.994667,0.011903
"""std""",9.089348,15.825105,32342.621782,18429.77294,117.806025,630.213372,250.227162,197.635576,250.227162,22.568677
"""min""",0.0,0.0,0.000013,120.0,42.0,46.0,46.0,0.0,46.0,-0.006466
"""25%""",5.36,64.0,9886.397171,6000.0,60.0,60.0,60.0,0.0,60.0,0.000028
"""50%""",10.04,64.0,23234.566807,6000.0,60.0,60.0,60.0,0.0,60.0,0.000044
"""75%""",20.0,64.0,36737.356574,6020.0,60.0,98.0,60.61,1.407053,60.61,0.000102
"""max""",60.0,255.0,1.572864e7,316492.0,7306.0,52194.0,9430.3,11655.404669,9430.3,78612.003899


Como todas son estadísticas están derivadas de la duración y del tráfico de una conexión, es posible que algunas no sean necesarias para un modelo. Será necesario un análisis más profundo para saber cuáles son signficativas y cuáles redundantes.

También observamos que algunas características como el "Rate" tendrán que ser escaladas ya se su media está en 27.490,94.